In [ ]:
import pandas as pd
import os
import numpy as np
import sklearn
import json
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, PreTrainedTokenizer, TrainingArguments, Trainer
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from dataclasses import dataclass, field
import transformers
from peft import (
    LoraConfig,
    get_peft_model,
    get_peft_model_state_dict,
)
import json

In [ ]:
current_dir = os.path.abspath('')
data_path = os.path.join(current_dir, '..', 'seqtrainer-data')
dataset_path = os.path.join(current_dir, '..', 'hpc', "hpc_datasets")
output_dir = os.path.join(current_dir, '..', 'dnabert-out')

In [ ]:
df = pd.read_csv(os.path.join(dataset_path, "dna_seq_target_300000.csv"))

In [ ]:
df

In [ ]:
@dataclass
class DataCollatorForSupervisedDataset(object):
    """Collate examples for supervised fine-tuning."""

    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances):
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=self.tokenizer.pad_token_id
        )
        labels = torch.tensor(labels, dtype=torch.float32)
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )


In [ ]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = preds.squeeze() 
    mse = ((preds - labels) ** 2).mean()
    return {"mse": mse, "rmse": mse**0.5}

In [ ]:
class SupervisedDataset(Dataset):
    """Dataset for supervised fine-tuning."""

    def __init__(self, sequences, tokenizer, labels):

        super(SupervisedDataset, self).__init__()
        output = tokenizer(
            sequences,
            return_tensors="pt",
            padding="longest",
            max_length=4096,
            truncation=True,
        )

        self.input_ids = output["input_ids"]
        self.attention_mask = output["attention_mask"]
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.num_labels = len(set(labels))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i):
        return dict(input_ids=self.input_ids[i], labels=self.labels[i])

In [ ]:
X = df['sequence']
y = np.log1p(np.array(df['target']))

In [ ]:
# train, test, val
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)
collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

In [ ]:
train_dataset = SupervisedDataset(tokenizer=tokenizer, sequences=X_train.tolist(), labels=y_train.tolist())
val_dataset = SupervisedDataset(tokenizer=tokenizer, sequences=X_val.tolist(), labels=y_val.tolist())
test_dataset = SupervisedDataset(tokenizer=tokenizer, sequences=X_test.tolist(), labels=y_test.tolist())

In [ ]:
target_modules = ["Wqkv", "dense", "gated_layers"]
def model_init(trial=None):
    base_model = AutoModelForSequenceClassification.from_pretrained(
        "zhihan1996/DNABERT-2-117M",
        num_labels=1,
        problem_type="regression",
        trust_remote_code=True
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_CLS",
        inference_mode=False,
    )

    return get_peft_model(base_model, lora_config)

def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "warmup_ratio": trial.suggest_float("warmup_ratio", 0.0, 0.2),
        "optimizer": trial.suggest_categorical("optimizer", ["adamw_torch", "adamw_hf", "adafactor"]),
    }


training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    logging_steps=100,
    gradient_accumulation_steps=2,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="rmse",
    greater_is_better=False,
)

trainer = transformers.Trainer(
                        model_init=model_init,
                        tokenizer=tokenizer,
                        args=training_args,
                        compute_metrics=compute_metrics,
                        train_dataset=train_dataset,
                        eval_dataset=val_dataset,
                        data_collator=collator)

best_trial = trainer.hyperparameter_search(
    direction="minimize",
    backend="optuna",
    hp_space=optuna_hp_space,
    n_trials=50,
)



In [ ]:
with open(os.path.join(os.path.join(output_dir, "best-model"), "best_hyperparameters.json"), "w") as f:
    json.dump(best_trial.hyperparameters, f, indent=2)

In [ ]:
for param_name, param_value in best_trial.hyperparameters.items():
    setattr(trainer.args, param_name, param_value)

trainer.train()

trainer.save_model(os.path.join(output_dir, "best-model"))
trainer.tokenizer.save_pretrained(os.path.join(output_dir, "best-model"))
print(f"Best model and tokenizer saved to {os.path.join(output_dir, "best-model")}")

In [ ]:
results_path = os.path.join(os.path.join(output_dir, "best-model"), "results")
results = trainer.evaluate(eval_dataset=test_dataset)
os.makedirs(results_path, exist_ok=True)
with open(os.path.join(results_path, "eval_results.json"), "w") as f:
    json.dump(results, f)

In [ ]:
#class DNABertTuned(torch.nn.Module):
#     def __init__(self):
#         super(DNABertTuned, self).__init__()
#         # self.regression = nn.Linear(768, 1)
#         self.bert = AutoModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)
#         self.regression_head = torch.nn.Linear(self.bert.config.hidden_size, 1) 

#     def forward(self, input_ids, attention_mask=None):
#         outputs = self.bert(input_ids, attention_mask=attention_mask)[0]
#         pooled_output = torch.mean(outputs, dim=1)
#         regression_output = self.regression_head(pooled_output)
#         return regression_output.squeeze(-1)

# model = DNABertTuned()
# model.train()

# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

# seqs = list(df['sequence'])
# inputs = tokenizer(seqs, return_tensors = 'pt', padding=True)
# output = model(inputs['input_ids'], attention_mask=inputs['attention_mask'])
# print(output.shape)
# print(target.shape)
# loss = torch.nn.MSELoss(target, output)
# optimizer.zero_grad()
# loss.backward()
# optimizer.step()
